<a href="https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("Hugging Face authentication is ready.")

Hugging Face authentication is ready.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [12]:
query_features = """
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(ga4_sessions) AS sessions,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,

    AVG(gsc_avg_position) AS avg_position

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE ga4_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(query_features).df()

print("Shape:", features.shape)
display(features.head(10))

print("\nSummary statistics:")
display(
    features[
        ["impressions", "clicks", "sessions", "ctr", "avg_position"]
    ].describe()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (90489, 7)


,client_hash_id,content_hash_id,impressions,clicks,sessions,ctr,avg_position
0,client_9958f0a7ae1df715,content_810cf06597918291,257.0,1.0,42.0,0.389105,11.186123
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,180.0,1.0,7.0,0.555556,8.674734
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,19657.0,199.0,168.0,1.012362,4.532382
3,client_9958f0a7ae1df715,content_f5e11209b398d173,47.0,0.0,11.0,0.000000,10.045000
4,client_9958f0a7ae1df715,content_8f3fa2db89105948,1.0,0.0,2.0,0.000000,7.000000
5,client_9958f0a7ae1df715,content_278030b007943b07,319.0,7.0,16.0,2.194357,5.914484
6,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,6106.0,39.0,40.0,0.638716,7.108246
7,client_9958f0a7ae1df715,content_347fbafb77d3ae37,396.0,4.0,38.0,1.010101,21.671343
8,client_9958f0a7ae1df715,content_15bd72d24e0a0b08,428.0,2.0,15.0,0.467290,13.669138
9,client_9958f0a7ae1df715,content_6f4cc70af7be346e,133.0,5.0,11.0,3.759398,9.234436



Summary statistics:


,impressions,clicks,sessions,ctr,avg_position
count,90489.000000,90489.000000,90489.000000,63856.000000,63856.000000
mean,943.061809,4.367426,14.364265,3.087267,13.374350
std,4701.289398,27.934660,47.007411,12.115273,13.939442
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,0.000000,4.321690
50%,24.000000,0.000000,2.000000,0.341297,8.000000
75%,304.000000,2.000000,8.000000,1.336347,18.350304
max,617124.000000,5668.000000,2730.000000,100.000000,305.500000


In [13]:
features["position_bucket"] = pd.cut(
    features["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["Top 3", "Page 1", "Page 2", "Deep"]
)

print("Position buckets:")
display(
    features["position_bucket"]
    .value_counts(dropna=False)
    .rename_axis("position_bucket")
    .reset_index(name="n")
)

Position buckets:


,position_bucket,n
0,Page 1,28002
1,NaN,27774
2,Deep,14443
3,Page 2,12179
4,Top 3,8091


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [14]:
# SIGNAL 1: CTR versus average position

ctr_position_table = (
    features
    .dropna(subset=["ctr", "avg_position"])
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("Signal 1 — CTR vs position")
display(ctr_position_table)

ctr_values = ctr_position_table["median_ctr"].dropna().tolist()

if len(ctr_values) >= 2:
    if all(
        ctr_values[i] >= ctr_values[i + 1]
        for i in range(len(ctr_values) - 1)
    ):
        signal_1_verdict = "CONFIRMED"
    elif all(
        ctr_values[i] <= ctr_values[i + 1]
        for i in range(len(ctr_values) - 1)
    ):
        signal_1_verdict = "OPPOSITE"
    else:
        signal_1_verdict = "MIXED"
else:
    signal_1_verdict = "FALSE"

print("Signal 1 verdict:", signal_1_verdict)

Signal 1 — CTR vs position


,position_bucket,n,median_ctr,median_impressions,median_position
0,Top 3,8091,0.638978,70.0,2.050099
1,Page 1,28002,0.558659,122.0,5.754330
2,Page 2,12179,0.381492,127.0,14.169322
3,Deep,14443,0.000000,112.0,30.000000


Signal 1 verdict: CONFIRMED


In [15]:
# SIGNAL 2: Search volume / impressions

volume_bins = [-1, 99, 499, 999, 4999, np.inf]
volume_labels = [
    "<100",
    "100-499",
    "500-999",
    "1000-4999",
    "5000+"
]

features["impression_bucket"] = pd.cut(
    features["impressions"],
    bins=volume_bins,
    labels=volume_labels
)

volume_table = (
    features
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr", "median"),
        median_clicks=("clicks", "median"),
        median_sessions=("sessions", "median")
    )
    .reset_index()
)

print("Signal 2 — Search volume")
display(volume_table)

print("\nSignal 2 verdict: CONFIRMED")

Signal 2 — Search volume


,impression_bucket,n,median_ctr,median_clicks,median_sessions
0,<100,57893,0.000000,0.0,1.0
1,100-499,14044,0.729927,2.0,5.0
2,500-999,5272,0.531091,4.0,11.0
3,1000-4999,9253,0.413052,8.0,24.0
4,5000+,4027,0.305281,30.0,67.0



Signal 2 verdict: CONFIRMED


In [16]:
# SIGNAL 3: Click availability with search exposure

click_table = (
    features
    .assign(
        has_clicks=features["clicks"] > 0
    )
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        pages_with_clicks=("has_clicks", "sum"),
        median_clicks=("clicks", "median")
    )
    .reset_index()
)

click_table["click_page_rate_pct"] = (
    click_table["pages_with_clicks"]
    / click_table["n"]
    * 100
)

print("Signal 3 — Click availability")
display(click_table)

print("\nSignal 3 verdict: MIXED")

Signal 3 — Click availability


,impression_bucket,n,pages_with_clicks,median_clicks,click_page_rate_pct
0,<100,57893,11737,0.0,20.273608
1,100-499,14044,10447,2.0,74.387639
2,500-999,5272,4567,4.0,86.627466
3,1000-4999,9253,8706,8.0,94.088404
4,5000+,4027,3982,30.0,98.882543



Signal 3 verdict: MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [17]:
# FLAG-LINKED TEST
# FlyRank's CTR-fix logic uses CTR together with search visibility/position.

flag_test = (
    features
    .dropna(subset=["ctr", "avg_position"])
    .assign(
        visible_page=(
            (features["avg_position"] > 0) &
            (features["avg_position"] <= 20)
        ),
        low_ctr=features["ctr"] < 0.5
    )
)

flag_test["ctr_review_candidate"] = (
    flag_test["visible_page"] &
    flag_test["low_ctr"] &
    (flag_test["impressions"] >= 500)
)

flag_table = (
    flag_test
    .groupby("ctr_review_candidate")
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("Flag-linked test — CTR review context")
display(flag_table)

flag_count = int(flag_test["ctr_review_candidate"].sum())

print("\nRows matching the CTR review context:", flag_count)

if flag_count > 0:
    print("Flag-linked verdict: CONFIRMED")
else:
    print("Flag-linked verdict: FALSE")

Flag-linked test — CTR review context


,ctr_review_candidate,n,median_impressions,median_ctr,median_position
0,False,57278,74.0,0.450450,8.279928
1,True,6578,1996.5,0.270514,6.007009



Rows matching the CTR review context: 6578
Flag-linked verdict: CONFIRMED


## 4. What this means in practice

### Practical takeaway

The signal checks suggest that search visibility and CTR should be interpreted together rather than separately. Pages with meaningful impressions provide stronger evidence than very low-volume pages. Therefore, the baseline will prioritize pages that have enough search exposure and whose CTR is weak relative to their observed position.

This is a directional decision-support rule for review, not proof that changing a page will improve its performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.